# Demo Langchain SQL Assistant with Langchain on Teradata Vantage

In [ ]:
import os
import sys
from pathlib import Path
from pprint import pprint
from dotenv import load_dotenv

# Load API key safely from .env
load_dotenv()

from agent_builder import create_sql_agent_with_mcp
from agent_ui import build_skill_answer, launch_agent_ui

import langchain
print(langchain.__version__)

## 1. Connect a LLM through API

In [ ]:
from langchain_openai import ChatOpenAI

if True:
    MODEL = ChatOpenAI(
        model="mistralai/Ministral-3-14B-Instruct-2512",
        base_url="https://api-dmproject.myddns.me/v1",
        api_key=os.environ["LOCAL_TOKEN"],  # or your mapped env var like FS_API_KEY, LOCAL_TOKEN
        temperature=0,
    )
else:
    MODEL = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 2. Specify a Teradata MCP Server

In [ ]:
MCP_SERVERS = {
    "teradata": {
        "transport": "http",  
        "url": "https://dmproject.myddns.me/mcpce/",
        
        # Uncomment ONLY if your MCP server requires auth
        # "headers": {
        #     "Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}",
        # }
    }
}

## 3. Instanciate a lanchain agent with skills and MCP server

In [ ]:
agent = await create_sql_agent_with_mcp(
    model=MODEL,
    skills_root=Path("./skills"),
    mcp_servers=MCP_SERVERS,
    system_prompt="""
You are a SQL assistant.

Rules:
- When a request matches a skill, load the skill first.
- Then use MCP tools to interact with the database.
- Do NOT invent schema.
- Be concise and correct.
""",
)

## 4. Build a chatbot with gradio

In [ ]:
skill_answer = build_skill_answer(agent=agent, return_trace=True)

In [ ]:
async def chat(message, history):
    answer, trace_md = await skill_answer(message)
    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": answer},
    ]
    return history, trace_md, ""

demo = launch_agent_ui(
    chat_fn=chat,
    title="SQL Assistant (Skills + MCP)",
    port=7860,
)

In [ ]:
demo.close()